In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

CATALOG = "workspace"
SCHEMA = "oss_ecosystem"

SILVER_CORE_TABLE = f"{CATALOG}.{SCHEMA}.silver_events_core"

GOLD_REPO_DAILY_TABLE = (
    f"{CATALOG}.{SCHEMA}.gold_repo_daily_activity"
)

In [0]:
repo_daily_base_df = (
    spark.table(SILVER_CORE_TABLE)
    .filter(F.col("repo_id").isNotNull())
    .withColumn(
        "event_date",
        F.to_date("event_ts")
    )
    .groupBy(
        "repo_id",
        "event_date"
    )
    .agg(
        F.max_by("repo_name", "event_ts")
            .alias("repo_name"),

        F.count("*")
            .alias("total_events"),

        F.countDistinct("actor_id")
            .alias("active_contributors"),

        F.sum(
            (F.col("event_type") == "PushEvent").cast("int")
        ).alias("push_events"),

        F.sum(
            (F.col("event_type") == "PullRequestEvent").cast("int")
        ).alias("pull_request_events"),

        F.sum(
            (F.col("event_type") == "IssuesEvent").cast("int")
        ).alias("issue_events"),

        F.sum(
            (F.col("event_type") == "IssueCommentEvent").cast("int")
        ).alias("issue_comment_events"),

        F.sum(
            (F.col("event_type") == "WatchEvent").cast("int")
        ).alias("watch_events"),

        F.sum(
            (F.col("event_type") == "ForkEvent").cast("int")
        ).alias("fork_events"),

        F.sum(
            (F.col("event_type") == "ReleaseEvent").cast("int")
        ).alias("release_events")
    )
)

In [0]:
print(
    "Gold daily rows:",
    repo_daily_base_df.count()
)

repo_daily_base_df.orderBy(
    F.desc("total_events")
).show(
    20,
    truncate=False
)

In [0]:
repo_daily_base_df.select(
    F.sum(
        F.col("repo_id").isNull().cast("int")
    ).alias("null_repo_id"),

    F.sum(
        F.col("event_date").isNull().cast("int")
    ).alias("null_event_date"),

    F.sum(
        (F.col("total_events") <= 0).cast("int")
    ).alias("invalid_total_events"),

    F.sum(
        (F.col("active_contributors") <= 0).cast("int")
    ).alias("invalid_active_contributors")
).show()

In [0]:
duplicate_gold_keys = (
    repo_daily_base_df
    .groupBy(
        "repo_id",
        "event_date"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Duplicate repo/date keys:",
    duplicate_gold_keys
)

In [0]:
SILVER_PR_TABLE = f"{CATALOG}.{SCHEMA}.silver_pull_request_events"
SILVER_ISSUE_TABLE = f"{CATALOG}.{SCHEMA}.silver_issue_events"
SILVER_WATCH_TABLE = f"{CATALOG}.{SCHEMA}.silver_watch_events"
SILVER_FORK_TABLE = f"{CATALOG}.{SCHEMA}.silver_fork_events"

pr_df = spark.table(SILVER_PR_TABLE)
issue_df = spark.table(SILVER_ISSUE_TABLE)
watch_df = spark.table(SILVER_WATCH_TABLE)
fork_df = spark.table(SILVER_FORK_TABLE)

In [0]:
pr_daily_df = (
    pr_df
    .withColumn("event_date", F.to_date("event_ts"))
    .groupBy("repo_id", "event_date")
    .agg(
        F.sum(
            (F.col("action") == "opened").cast("int")
        ).alias("prs_opened"),

        F.sum(
            (F.col("action") == "closed").cast("int")
        ).alias("prs_closed")
    )
)

In [0]:
issue_daily_df = (
    issue_df
    .withColumn("event_date", F.to_date("event_ts"))
    .groupBy("repo_id", "event_date")
    .agg(
        F.sum(
            (F.col("action") == "opened").cast("int")
        ).alias("issues_opened"),

        F.sum(
            (F.col("action") == "closed").cast("int")
        ).alias("issues_closed")
    )
)

In [0]:
watch_daily_df = (
    watch_df
    .withColumn("event_date", F.to_date("event_ts"))
    .groupBy("repo_id", "event_date")
    .agg(
        F.count("*").alias("stars")
    )
)

fork_daily_df = (
    fork_df
    .filter(F.col("repo_id").isNotNull())
    .withColumn("event_date", F.to_date("event_ts"))
    .groupBy("repo_id", "event_date")
    .agg(
        F.count("*").alias("forks")
    )
)

In [0]:
gold_repo_daily_df = (
    repo_daily_base_df
    .join(
        pr_daily_df,
        on=["repo_id", "event_date"],
        how="left"
    )
    .join(
        issue_daily_df,
        on=["repo_id", "event_date"],
        how="left"
    )
    .join(
        watch_daily_df,
        on=["repo_id", "event_date"],
        how="left"
    )
    .join(
        fork_daily_df,
        on=["repo_id", "event_date"],
        how="left"
    )
    .fillna(
        0,
        subset=[
            "prs_opened",
            "prs_closed",
            "issues_opened",
            "issues_closed",
            "stars",
            "forks"
        ]
    )
)

In [0]:
print(
    "Enriched Gold rows:",
    gold_repo_daily_df.count()
)

duplicate_gold_keys = (
    gold_repo_daily_df
    .groupBy("repo_id", "event_date")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Duplicate repo/date keys:",
    duplicate_gold_keys
)

gold_repo_daily_df.select(
    F.sum("prs_opened").alias("total_prs_opened"),
    F.sum("prs_closed").alias("total_prs_closed"),
    F.sum("issues_opened").alias("total_issues_opened"),
    F.sum("issues_closed").alias("total_issues_closed"),
    F.sum("stars").alias("total_stars"),
    F.sum("forks").alias("total_forks")
).show()

In [0]:
(
    gold_repo_daily_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_REPO_DAILY_TABLE)
)

In [0]:
persisted_gold_df = spark.table(GOLD_REPO_DAILY_TABLE)

print(
    "Persisted Gold rows:",
    persisted_gold_df.count()
)

persisted_duplicate_keys = (
    persisted_gold_df
    .groupBy("repo_id", "event_date")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Persisted duplicate repo/date keys:",
    persisted_duplicate_keys
)

In [0]:
contributor_daily_activity_df = (
    spark.table(SILVER_CORE_TABLE)
    .filter(
        F.col("repo_id").isNotNull() &
        F.col("actor_id").isNotNull()
    )
    .withColumn(
        "event_date",
        F.to_date("event_ts")
    )
    .groupBy(
        "repo_id",
        "event_date",
        "actor_id"
    )
    .agg(
        F.count("*").alias("contributor_events")
    )
)

In [0]:
contributor_concentration_df = (
    contributor_daily_activity_df
    .groupBy(
        "repo_id",
        "event_date"
    )
    .agg(
        F.sum("contributor_events")
            .alias("contributor_event_total"),

        F.max("contributor_events")
            .alias("top_contributor_events")
    )
    .withColumn(
        "top_contributor_share",
        F.col("top_contributor_events") /
        F.col("contributor_event_total")
    )
)

In [0]:
gold_repo_daily_df = (
    gold_repo_daily_df
    .join(
        contributor_concentration_df.select(
            "repo_id",
            "event_date",
            "top_contributor_share"
        ),
        on=["repo_id", "event_date"],
        how="left"
    )
)

In [0]:
gold_repo_daily_df.select(
    F.min("top_contributor_share").alias("min_share"),
    F.expr(
        "percentile_approx(top_contributor_share, 0.5)"
    ).alias("median_share"),
    F.expr(
        "percentile_approx(top_contributor_share, 0.95)"
    ).alias("p95_share"),
    F.max("top_contributor_share").alias("max_share"),
    F.sum(
        F.col("top_contributor_share").isNull().cast("int")
    ).alias("null_share")
).show()

In [0]:
(
    gold_repo_daily_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_REPO_DAILY_TABLE)
)

In [0]:
final_gold_df = spark.table(GOLD_REPO_DAILY_TABLE)

print("Final Gold rows:", final_gold_df.count())

final_gold_df.select(
    F.sum(
        F.col("top_contributor_share").isNull().cast("int")
    ).alias("null_concentration")
).show()

final_duplicate_keys = (
    final_gold_df
    .groupBy("repo_id", "event_date")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Final duplicate repo/date keys:", final_duplicate_keys)

## Phase 10 conclusions

The first Gold analytical table, `gold_repo_daily_activity`, was created at a strict repository-day grain:

`repo_id + event_date`

Validation results:

- 73,532 repository-day rows
- 0 duplicate repository-day keys
- 0 null concentration values
- Repository activity and active contributor counts were retained
- Daily Push, Pull Request, Issue, Issue Comment, Watch, Fork, and Release event counts were included
- Pull request opened/closed and issue opened/closed metrics were derived from event-specific Silver tables
- Watch events were represented as daily star events
- Fork events were represented as daily fork events
- Contributor concentration was represented by the share of daily repository events generated by the most active contributor

One ForkEvent with a null source repository ID is intentionally excluded from repository-level Gold metrics.

New-versus-returning contributor metrics are deferred until the dataset spans multiple calendar days, because they are not analytically meaningful with the current partial-day dataset.